In [ ]:
import sys
from pathlib import Path

REPO = Path("../")
sys.path.append(str(REPO))

import torch
import torchvision
import matplotlib.pyplot as plt
from hydra.utils import instantiate
from hydra.core.hydra_config import HydraConfig
from hydra import compose, initialize
from hydra.core.hydra_config import HydraConfig

from logdiff.score.pipelines import CondDDIMPipeline
from logdiff.score.sampling_compositional import LogicModelWrapper, And, Or_CI, Or_ME, Not
from logdiff.score.sampling_compositional_constant import NotConstant, AndConstant, OrConstant
from logdiff.score.sampling_cmnist import Color, Digit
import torch
from logdiff.utils import set_seed
from logdiff.evaluate.evaluation_utils import get_null_token, load_models


In [ ]:
CONFIG_PATH = "../configs/"
CONFIG_NAME = "cmnist_inference"

with initialize(version_base="1.2", config_path=CONFIG_PATH):
    cfg = compose(config_name=CONFIG_NAME, return_hydra_config=True)

cfg.hydra.runtime.cwd = "./../"
HydraConfig.instance().set_config(cfg)

cfg.composition_classifier_checkpoint = REPO / cfg.composition_classifier_checkpoint
cfg.judge_classifier_checkpoint = REPO / cfg.judge_classifier_checkpoint
cfg.checkpoint_path = REPO / cfg.checkpoint_path

set_seed(cfg["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

########## Load model #############
scheduler = instantiate(cfg.noise_scheduler)
model, judge_classifier, composition_classifier = load_models(cfg, device)

########## Setup Pipeline #############
expr_wrapper = LogicModelWrapper(model, composition_classifier, False)
pipe = CondDDIMPipeline(net=expr_wrapper, scheduler=scheduler)

########## Helpers #############
def show(imgs, save=False):
    # Ensure images are on CPU and clipped correctly
    imgs = imgs.detach().cpu().clamp(0, 1)
    
    # Grid creation: This stitches images together into one big image
    # padding=0 ensures no black/white lines between images
    grid_img = torchvision.utils.make_grid(
        imgs, 
        nrow=int(len(imgs)**0.5), 
        padding=4,      # Adjust this number to make the gap wider/thinner
        pad_value=1     # 1 = White spacer, 0 = Black spacer
    )
    
    # Plotting
    plt.figure(figsize=(8, 8)) # Size doesn't matter much for PDF, just aspect ratio
    plt.imshow(grid_img.permute(1, 2, 0))
    plt.axis("off")
    
    # Removes all margins around the figure itself
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0) 
    
    if save:
        # bbox_inches='tight' and pad_inches=0 are critical for removing the white frame
        plt.savefig("../images/cmnist.pdf", bbox_inches='tight', pad_inches=0)
        
    plt.show()


In [ ]:
guidance = cfg.get("guidance", None)
guidance["atom"] = 2.5

In [ ]:
batch_size = 9
null_token = get_null_token(cfg, batch_size, device)

torch.manual_seed(1)

query = Not(Color("yellow"))
print(query)

images = pipe(
            batch_size=batch_size,
            num_inference_steps=100,
            guidance_dict=guidance,
            return_dict=True,
            null_token=null_token,
            query=query,
            use_clipped_model_output=True,
        ).images

show(images, save=True)


In [ ]:
#batch_size = 9
#null_token = get_null_token(cfg, batch_size, device)

#torch.manual_seed(3)

query = OrConstant(Digit(1), Color("pink"))
print(query)

images = pipe(
            batch_size=batch_size,
            num_inference_steps=100,
            guidance_dict=guidance,
            return_dict=True,
            null_token=null_token,
            query=query,
            use_clipped_model_output=True,
        ).images

show(images, save=True)


In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision.utils import make_grid
from torchvision.transforms.functional import to_pil_image

# 1. Setup Cache Directory
cache_dir = "figures/cache"
os.makedirs(cache_dir, exist_ok=True)

scales = [1.0, 1.5, 2.0, 2.5]
methods = ["Ours", "Constant"]
results = {method: [] for method in methods}

# Common setup
batch_size = 9 
null_token = get_null_token(cfg, batch_size, device)

# --- LOOP ---
for method in methods:
    for scale in scales:
        filename = os.path.join(cache_dir, f"smnist{method}_{scale}_grid.png")
        
        if os.path.exists(filename):
            print(f"Loading cached grid: {filename}")
            img_pil = Image.open(filename).convert("RGB")
            results[method].append(img_pil)
            
        else:
            print(f"Generating for Method: {method}, Scale: {scale}")
            
            torch.manual_seed(9) # Reset seed for fair comparison
            
            current_guidance = cfg.get("guidance", {}).copy()
            current_guidance["atom"] = scale
            
            if method == "Ours":
                query = Not(Digit(1)) 
            else:
                query = NotConstant(Digit(1))
            
            output_images = pipe(
                batch_size=batch_size,
                num_inference_steps=100,
                guidance_dict=current_guidance,
                return_dict=True,
                null_token=null_token,
                query=query,
                use_clipped_model_output=True,
            ).images
            
            # --- CREATE 3x3 GRID ---
            # output_images is likely [9, C, H, W]
            # make_grid stitches them into one large [C, H_grid, W_grid] tensor
            # nrow=3 ensures a 3x3 layout for a batch of 9
            grid_tensor = make_grid(output_images, nrow=3, padding=2, normalize=True)
            
            # Move to CPU and convert to PIL
            img_pil = to_pil_image(grid_tensor.cpu())
            
            img_pil.save(filename)
            results[method].append(img_pil)



In [ ]:
fig, axes = plt.subplots(nrows=len(methods), ncols=len(scales), figsize=(12, 6))
plt.subplots_adjust(wspace=0.1, hspace=0.3)

for row_idx, method in enumerate(methods):
    for col_idx, scale in enumerate(scales):
        ax = axes[row_idx, col_idx]
        img = results[method][col_idx]
        
        ax.imshow(img)
        ax.axis("off")
        
        if row_idx == 0:
            ax.set_title(f"$w = {scale}$", fontsize=14, fontweight='bold')
        if col_idx == 0:
            ax.text(-0.2, 0.5, method, transform=ax.transAxes, 
                    fontsize=14, fontweight='bold', va='center', rotation=90)

plt.tight_layout()
plt.savefig("../images/cmnist_guidance_effect_qualitative.pdf", bbox_inches='tight', dpi=300)
plt.show()